# Xarray-Spatial Morphology: Erosion, dilation, opening, and closing

Morphological operators filter rasters by sliding a structuring element (kernel) across the surface and picking the local minimum or maximum at each cell. They show up everywhere from cleaning noisy classification masks to smoothing elevation surfaces before further analysis. This notebook walks through the four operations in `xrspatial.morphology` on both continuous terrain and binary masks.

### What you'll build

1. Generate synthetic terrain and a hillshade base layer
2. Apply erosion and dilation to see how local min/max reshape the surface
3. Use opening and closing to selectively remove noise
4. Clean up a noisy binary classification mask
5. Compare square and circular structuring elements

![Morphological operators preview](images/morphological_operators_preview.png)

[Erosion and dilation](#Erosion-and-dilation) · [Opening and closing](#Opening-and-closing) · [Binary mask cleanup](#Binary-mask-cleanup) · [Circular structuring element](#Circular-structuring-element)

Standard imports plus the morphology submodule.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

import xrspatial
from xrspatial.morphology import (
    _circle_kernel,
    morph_closing,
    morph_dilate,
    morph_erode,
    morph_opening,
)

## Terrain data

Synthetic elevation built from overlapping Gaussians with added noise. The same raster is reused in every section below.

In [ ]:
W = 800
H = 600
x_range = (-20e6, 20e6)
y_range = (-20e6, 20e6)

terrain = xr.DataArray(np.zeros((H, W)))
terrain = terrain.xrs.generate_terrain(x_range=x_range, y_range=y_range)
illuminated = terrain.xrs.hillshade()

kernel = np.ones((5, 5), dtype=np.uint8)

terrain.plot.imshow(cmap='terrain', size=7.5, aspect=W/H, add_colorbar=False)

Blues are low areas, greens and browns climb to ridges and peaks. The terrain has enough variety to show how morphological operators reshape the surface at different scales.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
terrain.plot.imshow(ax=ax, cmap='terrain', alpha=128/255, add_colorbar=False)
ax.set_axis_off()

## Erosion and dilation

[Erosion](https://en.wikipedia.org/wiki/Erosion_(morphology)) replaces each cell with the minimum value inside the kernel footprint. Bright features shrink and valleys widen. [Dilation](https://en.wikipedia.org/wiki/Dilation_(morphology)) does the opposite: it takes the local maximum, so bright features expand and valleys fill in. Both use a 5x5 square kernel here.

The three panels share the same color scale so you can compare values directly.

In [ ]:
eroded = morph_erode(terrain, kernel=kernel, boundary='nearest')
dilated = morph_dilate(terrain, kernel=kernel, boundary='nearest')

vmin, vmax = float(terrain.min()), float(terrain.max())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, data, label in zip(
    axes,
    [terrain, eroded, dilated],
    ['Original', 'Eroded (local min)', 'Dilated (local max)'],
):
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    data.plot.imshow(ax=ax, cmap='terrain', alpha=160/255,
                     vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
plt.tight_layout()

Erosion pulled every surface downward toward local minima, while dilation pushed it up. Notice how ridgelines get thinner after erosion and valleys get shallower after dilation.

<div class="alert alert-block alert-warning">
<b>Edge handling.</b> The default <code>boundary='nan'</code> treats pixels outside the raster as NaN, which propagates inward by the kernel radius. Use <code>boundary='nearest'</code> (as above) to repeat edge values instead, or <code>'reflect'</code> to mirror them. The choice matters most when your area of interest extends to the raster edge.
</div>

## Opening and closing

[Opening](https://en.wikipedia.org/wiki/Opening_(morphology)) is erosion followed by dilation. It removes small bright features (spikes, noise) while leaving larger structures roughly intact. [Closing](https://en.wikipedia.org/wiki/Closing_(morphology)) is dilation followed by erosion, which fills small dark pits without inflating large features.

The three panels show the original terrain next to the opened and closed versions, all on the same color scale.

In [ ]:
opened = morph_opening(terrain, kernel=kernel, boundary='nearest')
closed = morph_closing(terrain, kernel=kernel, boundary='nearest')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, data, label in zip(
    axes,
    [terrain, opened, closed],
    ['Original', 'Opening (erode then dilate)', 'Closing (dilate then erode)'],
):
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    data.plot.imshow(ax=ax, cmap='terrain', alpha=160/255,
                     vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
plt.tight_layout()

Opening shaved off narrow peaks while keeping broad ridges. Closing filled in small depressions without raising the overall surface. The effects are subtle on smooth terrain but dramatic on noisy data, as the next section shows.

## Binary mask cleanup

The most common real-world use of morphological operators is cleaning up classification results. Salt noise (isolated bright pixels in dark regions) and pepper noise (isolated dark pixels in bright regions) are typical artifacts from pixel-level classifiers. Closing fills the pepper holes first, then opening removes the salt specks.

The two panels below show a noisy binary mask before and after a close-then-open pass.

In [ ]:
rng = np.random.default_rng(42)

# Build a binary mask with a large square region
mask = np.zeros((200, 200), dtype=np.float64)
mask[40:160, 40:160] = 1.0

# Sprinkle salt-and-pepper noise
noise = rng.random(mask.shape)
mask[noise < 0.02] = 1.0   # salt (bright specks in dark area)
mask[noise > 0.98] = 0.0   # pepper (dark specks in bright area)

noisy = xr.DataArray(mask, dims=['y', 'x'], name='mask')
cleaned = morph_opening(
    morph_closing(noisy, kernel=kernel, boundary='nearest'),
    kernel=kernel,
    boundary='nearest',
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
noisy.plot.imshow(ax=axes[0], cmap='gray', add_colorbar=False)
axes[0].set_title('Noisy mask', fontsize=13)
axes[0].set_axis_off()
cleaned.plot.imshow(ax=axes[1], cmap='gray', add_colorbar=False)
axes[1].set_title('After closing + opening', fontsize=13)
axes[1].set_axis_off()
plt.tight_layout()

<div class="alert alert-block alert-warning">
<b>Kernel size vs. feature size.</b> The kernel must be larger than the noise you want to remove. A 3x3 kernel only cleans isolated single-pixel noise. Clusters of 2-3 noisy pixels need a 5x5 or 7x7 kernel. But a larger kernel also eats into legitimate small features, so there is always a trade-off.
</div>

## Circular structuring element

A square kernel introduces directional bias along the diagonals because corner pixels are farther from the center than edge pixels. `_circle_kernel(radius)` builds a disk-shaped structuring element that treats all directions equally, which is usually a better default for natural features.

The plot compares erosion with a 5x5 square kernel against erosion with a circular kernel of radius 3 (giving a 7x7 footprint). Areas where the two results differ are highlighted in the overlay.

In [ ]:
disk = _circle_kernel(3)

eroded_square = morph_erode(terrain, kernel=kernel, boundary='nearest')
eroded_disk = morph_erode(terrain, kernel=disk, boundary='nearest')

# Difference: where the two kernels give different results
diff = np.abs(eroded_square.values - eroded_disk.values)
diff_mask = xr.DataArray(
    np.where(diff > 0.5, 1.0, np.nan),
    dims=['y', 'x'],
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

illuminated.plot.imshow(ax=axes[0], cmap='gray', add_colorbar=False)
eroded_square.plot.imshow(ax=axes[0], cmap='terrain', alpha=160/255,
                          vmin=vmin, vmax=vmax, add_colorbar=False)
axes[0].set_title('Eroded (5x5 square)', fontsize=13)
axes[0].set_axis_off()

illuminated.plot.imshow(ax=axes[1], cmap='gray', add_colorbar=False)
eroded_disk.plot.imshow(ax=axes[1], cmap='terrain', alpha=160/255,
                        vmin=vmin, vmax=vmax, add_colorbar=False)
axes[1].set_title('Eroded (circular r=3)', fontsize=13)
axes[1].set_axis_off()

illuminated.plot.imshow(ax=axes[2], cmap='gray', add_colorbar=False)
diff_mask.plot.imshow(ax=axes[2], cmap=ListedColormap(['darkorange']),
                      alpha=200/255, add_colorbar=False)
axes[2].legend(handles=[
    Patch(facecolor='darkorange', alpha=0.78, label='Difference > 0.5'),
], loc='lower right', fontsize=11, framealpha=0.9)
axes[2].set_title('Where they disagree', fontsize=13)
axes[2].set_axis_off()

plt.tight_layout()

The circular kernel produces a slightly deeper erosion (7x7 vs 5x5 footprint), and the orange highlights show where steep gradients amplify the shape difference. On gentle slopes the two kernels give nearly identical results.

In [ ]:
import os, matplotlib

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, data, label in zip(
    axes,
    [terrain, eroded, dilated, opened],
    ['Original', 'Eroded', 'Dilated', 'Opened'],
):
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    data.plot.imshow(ax=ax, cmap='terrain', alpha=160/255,
                     vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
plt.tight_layout()

os.makedirs('images', exist_ok=True)
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/morphological_operators_preview.png',
            bbox_inches='tight', dpi=120)
plt.close(fig)

### References

- [Erosion (morphology)](https://en.wikipedia.org/wiki/Erosion_(morphology)), Wikipedia
- [Dilation (morphology)](https://en.wikipedia.org/wiki/Dilation_(morphology)), Wikipedia
- [Opening (morphology)](https://en.wikipedia.org/wiki/Opening_(morphology)), Wikipedia
- [Closing (morphology)](https://en.wikipedia.org/wiki/Closing_(morphology)), Wikipedia
- [Mathematical morphology](https://en.wikipedia.org/wiki/Mathematical_morphology), Wikipedia
- [Structuring element](https://en.wikipedia.org/wiki/Structuring_element), Wikipedia